# Assistente medico via LangGraph (Step 6) no Google Colab

Este notebook roda o fluxo LangGraph do assistente medico no Colab. Usa o mesmo modelo fine-tunado (PEFT/QLoRA) do projeto.

Antes de rodar:
1. Ative a GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU.
2. O modelo fine-tunado deve estar em outputs/finetune_pqal (clone ou Drive).
3. Clone o repositorio na celula 2.

## 1. Verificar GPU e montar Drive (opcional)

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu.returncode != 0:
    print("Aviso: GPU nao detectada. Ative em Runtime -> Change runtime type -> GPU.")
else:
    print(gpu.stdout)

from google.colab import drive
drive.mount("/content/drive")

## 2. Clonar repositorio e instalar dependencias

In [ ]:
!git clone https://github.com/thiagonespereira/POSTECH-FIAP-FASE3.git /content/POSTECH-FIAP-FASE3
%cd /content/POSTECH-FIAP-FASE3
!pip install -q -r requirements.txt
print("Repositorio e dependencias prontos.")

## 3. Caminho do modelo e compilar o grafo

Ajuste MODEL_DIR se o modelo estiver no Drive.

In [ ]:
import sys
from pathlib import Path

REPO = Path("/content/POSTECH-FIAP-FASE3")
sys.path.insert(0, str(REPO))

MODEL_DIR = REPO / "outputs" / "finetune_pqal"
# MODEL_DIR = Path("/content/drive/MyDrive/POSTECH-FIAP-FASE3/outputs/finetune_pqal")

if not MODEL_DIR.exists():
    print("Aviso:", MODEL_DIR, "nao existe.")
else:
    print("Modelo:", MODEL_DIR)

In [ ]:
from src.graphs.medical_flow import build_medical_graph

print("Carregando modelo e compilando grafo...")
app = build_medical_graph(MODEL_DIR, max_new_tokens=256)
print("Grafo compilado.")

## 4. (Opcional) Ver estrutura do grafo em ASCII

O `draw_ascii()` precisa do pacote **grandalf** (já listado em `requirements.txt`). Se ainda der erro, rode: `!pip install -q grandalf`.

In [ ]:
# grandalf: necessário para draw_ascii() no LangGraph/LangChain
!pip install -q grandalf
print(app.get_graph().draw_ascii())

## 5. Perguntar ao assistente

In [ ]:
pergunta = "Does aspirin reduce fever?"
contexto = ""

estado_inicial = {"pergunta": pergunta, "contexto": contexto or ""}
resultado = app.invoke(estado_inicial)

print("--- Resposta ---")
print(resultado.get("resposta", ""))
print("\n--- Historico ---")
for i, h in enumerate(resultado.get("historico") or []):
    content = getattr(h, "content", h) if not isinstance(h, dict) else h.get("content", h)
    print(i+1, content)

## 6. Outro exemplo

In [ ]:
pergunta2 = "Can the PHQ-9 assess depression in people with vision loss?"
contexto2 = "The PHQ-9 was completed by 103 participants with low vision."
resultado2 = app.invoke({"pergunta": pergunta2, "contexto": contexto2})
print(resultado2.get("resposta", ""))